In [2]:
import tensorflow as tf
tf.random.set_seed(42)

In [3]:
import sys
sys.path.append("../src")

from data_loader import load_all
from lstm_utils import build_sequences
from fault_reference import HARD_TO_DETECT_FAULTS

train, test = load_all()

measurement_cols = [c for c in train.columns if c.startswith("XMEAS") or c.startswith("XMV")]

train_filtered = train[~train["faultNumber"].isin(HARD_TO_DETECT_FAULTS)]

X_train_seq, y_train_seq = build_sequences(train_filtered, measurement_cols, window=20)

print("X shape:", X_train_seq.shape)
print("y shape:", y_train_seq.shape)

X shape: (8779, 20, 52)
y shape: (8779,)


In [4]:
#test data
test_filtered = test[~test["faultNumber"].isin(HARD_TO_DETECT_FAULTS)]

X_test_seq, y_test_seq = build_sequences(test_filtered, measurement_cols, window=20)

print("X_test shape:", X_test_seq.shape)
print("y_test shape:", y_test_seq.shape)

X_test shape: (17879, 20, 52)
y_test shape: (17879,)


In [5]:
# standardizing and encodeing the fault labels. Standardization is necessary for lstm models to perform well. 
# The fault labels are encoded to integers for the model to predict.    

from sklearn.preprocessing import StandardScaler, LabelEncoder

n_samples, n_timesteps, n_features = X_train_seq.shape

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_seq.reshape(-1, n_features))
X_train_scaled = X_train_scaled.reshape(n_samples, n_timesteps, n_features)

n_test_samples = X_test_seq.shape[0]
X_test_scaled = scaler.transform(X_test_seq.reshape(-1, n_features))
X_test_scaled = X_test_scaled.reshape(n_test_samples, n_timesteps, n_features)

label_encoder_lstm = LabelEncoder()
y_train_encoded = label_encoder_lstm.fit_transform(y_train_seq)
y_test_encoded = label_encoder_lstm.transform(y_test_seq)

print("X_train_scaled shape:", X_train_scaled.shape)
print("Number of classes:", len(label_encoder_lstm.classes_))

X_train_scaled shape: (8779, 20, 52)
Number of classes: 19


In [6]:
# Building the LSTM model using Keras. The model consists of two LSTM layers \
# followed by a dense layer with softmax activation for multi-class classification.

from tensorflow import keras
from tensorflow.keras import layers

model_lstm = keras.Sequential([
    layers.Input(shape=(n_timesteps, n_features)),
    layers.LSTM(64),
    layers.Dense(19, activation="softmax")
])

model_lstm.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_lstm.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        29,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 19)             │         1,235 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,187 (121.82 KB)

 Trainable params: 31,187 (121.82 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
#Training the LSTM model using the training data. The model is trained for 20 epochs with a batch size of 64. 
# A validation split of 0.2 is used to monitor the model's performance on unseen data during training.

history = model_lstm.fit(
    X_train_scaled, y_train_encoded,
    validation_split=0.2,
    epochs=20,
    batch_size=64,
    verbose=1
)

Epoch 1/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 20s 70ms/step - accuracy: 0.5167 - loss: 1.7001 - val_accuracy: 0.0000e+00 - val_loss: 5.8834
Epoch 2/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 55ms/step - accuracy: 0.8576 - loss: 0.5135 - val_accuracy: 0.0188 - val_loss: 6.7899
Epoch 3/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 51ms/step - accuracy: 0.9459 - loss: 0.2155 - val_accuracy: 0.0222 - val_loss: 7.4500
Epoch 4/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 50ms/step - accuracy: 0.9775 - loss: 0.1103 - val_accuracy: 0.0228 - val_loss: 8.0309
Epoch 5/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 51ms/step - accuracy: 0.9869 - loss: 0.0687 - val_accuracy: 0.0228 - val_loss: 8.4246
Epoch 6/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 54ms/step - accuracy: 0.9966 - loss: 0.0314 - val_accuracy: 0.0233 - val_loss: 8.7544
Epoch 7/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - accuracy: 1.0000 - loss: 0.0148 - val_accuracy: 0.0239 - val_loss: 9.2080
Epoch 8/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 52ms/step - accuracy: 0.9996 - loss: 0.0113 - val

`validation_split=0.2` doesn't shuffle — it just takes the last 20% of
the array. Since sequences are built fault-by-fault in order, that
last 20% was mostly just the final few faults, which training barely
saw. Result: 100% train accuracy, ~0% validation (below random chance)
— a broken split, not real overfitting.

**Fix**: `train_test_split(..., stratify=...)` first, then pass
`validation_data=(...)` explicitly instead.

In [8]:
# Properly splitting the data and retraining

from sklearn.model_selection import train_test_split

X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_scaled, y_train_encoded,
    test_size=0.2,
    stratify=y_train_encoded,
    random_state=42
)

history = model_lstm.fit(
    X_train_split, y_train_split,
    validation_data=(X_val_split, y_val_split),
    epochs=20,
    batch_size=64,
    verbose=1
)

Epoch 1/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 7s 60ms/step - accuracy: 0.8229 - loss: 0.6452 - val_accuracy: 0.9322 - val_loss: 0.2288
Epoch 2/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 57ms/step - accuracy: 0.9600 - loss: 0.1510 - val_accuracy: 0.9875 - val_loss: 0.0764
Epoch 3/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 54ms/step - accuracy: 0.9829 - loss: 0.0722 - val_accuracy: 0.9875 - val_loss: 0.0601
Epoch 4/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 7s 64ms/step - accuracy: 0.9906 - loss: 0.0400 - val_accuracy: 0.9915 - val_loss: 0.0364
Epoch 5/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 8s 70ms/step - accuracy: 0.9954 - loss: 0.0224 - val_accuracy: 0.9983 - val_loss: 0.0154
Epoch 6/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 7s 64ms/step - accuracy: 0.9999 - loss: 0.0066 - val_accuracy: 0.9994 - val_loss: 0.0084
Epoch 7/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 7s 61ms/step - accuracy: 1.0000 - loss: 0.0041 - val_accuracy: 0.9989 - val_loss: 0.0084
Epoch 8/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 10s 59ms/step - accuracy: 1.0000 - loss: 0.0031 - val_acc

## Fixing data leakage from overlapping windows

99.77% validation accuracy was suspiciously high — likely because
`step=1` creates heavily overlapping sequences (95% shared rows
between adjacent windows). Random shuffling before splitting let
near-duplicate windows land in both train and validation, so the
model wasn't really being tested on unseen data.

**Fix**: split each fault's run by time first (80% early rows for
training, 20% later rows for validation), then build windows
separately within each portion — guaranteeing no shared rows between
train and validation.

In [9]:
import numpy as np
from sklearn.model_selection import train_test_split

def build_sequences_with_run_split(df, feature_cols, window=20, step=1, val_fraction=0.2, random_state=42):
    train_seqs_X, train_seqs_y = [], []
    val_seqs_X, val_seqs_y = [], []

    for fault_num in sorted(df["faultNumber"].unique()):
        fault_data = df[df["faultNumber"] == fault_num][feature_cols].values
        n_rows = len(fault_data)

        split_point = int(n_rows * (1 - val_fraction))

        train_part = fault_data[:split_point]
        val_part = fault_data[split_point:]

        for start in range(0, len(train_part) - window + 1, step):
            train_seqs_X.append(train_part[start:start + window])
            train_seqs_y.append(fault_num)

        for start in range(0, len(val_part) - window + 1, step):
            val_seqs_X.append(val_part[start:start + window])
            val_seqs_y.append(fault_num)

    return (np.array(train_seqs_X), np.array(train_seqs_y),
            np.array(val_seqs_X), np.array(val_seqs_y))


X_train_clean, y_train_clean, X_val_clean, y_val_clean = build_sequences_with_run_split(
    train_filtered, measurement_cols, window=20
)

print("Train:", X_train_clean.shape)
print("Val:", X_val_clean.shape)

Train: (6951, 20, 52)
Val: (1467, 20, 52)


In [10]:
#scaling
n_train_samples, n_timesteps, n_features = X_train_clean.shape
n_val_samples = X_val_clean.shape[0]

scaler_clean = StandardScaler()
X_train_clean_scaled = scaler_clean.fit_transform(X_train_clean.reshape(-1, n_features))
X_train_clean_scaled = X_train_clean_scaled.reshape(n_train_samples, n_timesteps, n_features)

X_val_clean_scaled = scaler_clean.transform(X_val_clean.reshape(-1, n_features))
X_val_clean_scaled = X_val_clean_scaled.reshape(n_val_samples, n_timesteps, n_features)

label_encoder_clean = LabelEncoder()
y_train_clean_encoded = label_encoder_clean.fit_transform(y_train_clean)
y_val_clean_encoded = label_encoder_clean.transform(y_val_clean)

print("Scaled and encoded successfully")

Scaled and encoded successfully


In [11]:
# Rebuilt the model

model_lstm_v2 = keras.Sequential([
    layers.Input(shape=(n_timesteps, n_features)),
    layers.LSTM(64),
    layers.Dense(19, activation="softmax")
])

model_lstm_v2.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_v2 = model_lstm_v2.fit(
    X_train_clean_scaled, y_train_clean_encoded,
    validation_data=(X_val_clean_scaled, y_val_clean_encoded),
    epochs=20,
    batch_size=64,
    verbose=1
)

Epoch 1/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 19s 71ms/step - accuracy: 0.4832 - loss: 1.9411 - val_accuracy: 0.6169 - val_loss: 1.3469
Epoch 2/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 6s 58ms/step - accuracy: 0.8443 - loss: 0.6262 - val_accuracy: 0.7464 - val_loss: 0.8596
Epoch 3/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - accuracy: 0.9530 - loss: 0.2298 - val_accuracy: 0.7860 - val_loss: 0.7806
Epoch 4/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 6s 51ms/step - accuracy: 0.9786 - loss: 0.1143 - val_accuracy: 0.7587 - val_loss: 0.8933
Epoch 5/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 6s 50ms/step - accuracy: 0.9915 - loss: 0.0619 - val_accuracy: 0.7860 - val_loss: 0.8487
Epoch 6/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 6s 51ms/step - accuracy: 0.9970 - loss: 0.0316 - val_accuracy: 0.7935 - val_loss: 0.8648
Epoch 7/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 6s 52ms/step - accuracy: 0.9973 - loss: 0.0247 - val_accuracy: 0.7989 - val_loss: 0.8538
Epoch 8/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 10s 51ms/step - accuracy: 1.0000 - loss: 0.0098 - val_ac

In [12]:
from tensorflow.keras.callbacks import EarlyStopping

model_lstm_v3 = keras.Sequential([
    layers.Input(shape=(n_timesteps, n_features)),
    layers.LSTM(64),
    layers.Dense(19, activation="softmax")
])

model_lstm_v3.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=3,
    restore_best_weights=True
)

history_v3 = model_lstm_v3.fit(
    X_train_clean_scaled, y_train_clean_encoded,
    validation_data=(X_val_clean_scaled, y_val_clean_encoded),
    epochs=20,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 15s 62ms/step - accuracy: 0.4935 - loss: 1.9152 - val_accuracy: 0.5890 - val_loss: 1.3693
Epoch 2/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 10s 55ms/step - accuracy: 0.8561 - loss: 0.6129 - val_accuracy: 0.7171 - val_loss: 1.0350
Epoch 3/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - accuracy: 0.9478 - loss: 0.2447 - val_accuracy: 0.7601 - val_loss: 0.9332
Epoch 4/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - accuracy: 0.9801 - loss: 0.1104 - val_accuracy: 0.7791 - val_loss: 0.9283
Epoch 5/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - accuracy: 0.9922 - loss: 0.0620 - val_accuracy: 0.7969 - val_loss: 0.9415
Epoch 6/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - accuracy: 0.9964 - loss: 0.0383 - val_accuracy: 0.7948 - val_loss: 0.9525
Epoch 7/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - accuracy: 0.9965 - loss: 0.0292 - val_accuracy: 0.8037 - val_loss: 0.8952
Epoch 8/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 7s 59ms/step - accuracy: 0.9958 - loss: 0.0302 - val_ac

In [13]:
#evaluation

n_test_samples = X_test_seq.shape[0]

X_test_seq_scaled = scaler_clean.transform(X_test_seq.reshape(-1, n_features))
X_test_seq_scaled = X_test_seq_scaled.reshape(n_test_samples, n_timesteps, n_features)

y_test_seq_encoded = label_encoder_clean.transform(y_test_seq)

test_loss, test_accuracy = model_lstm_v3.evaluate(X_test_seq_scaled, y_test_seq_encoded, verbose=0)

print(f"LSTM test accuracy: {test_accuracy:.2%}")
#print(f"XGBoost (tuned + trend) test accuracy: {accuracy_trend:.2%}")

LSTM test accuracy: 60.01%


In [14]:
y_pred_lstm_encoded = model_lstm_v3.predict(X_test_seq_scaled, verbose=0).argmax(axis=1)
y_pred_lstm = label_encoder_clean.inverse_transform(y_pred_lstm_encoded)

# Reconstruct which "starting sample" each test sequence came from, per fault
sample_positions = []
for fault_num in sorted(test_filtered["faultNumber"].unique()):
    n_rows = len(test_filtered[test_filtered["faultNumber"] == fault_num])
    n_windows = n_rows - 20 + 1
    sample_positions.extend(range(1, n_windows + 1))

sample_positions = np.array(sample_positions)

# Check accuracy specifically for windows starting before vs after sample 160
early_mask = sample_positions <= 140  # windows fully inside the pre-fault period
late_mask = sample_positions > 160

early_correct = (y_pred_lstm[early_mask] == y_test_seq[early_mask]).mean()
late_correct = (y_pred_lstm[late_mask] == y_test_seq[late_mask]).mean()

print(f"Accuracy on early (pre-fault, mislabeled) windows: {early_correct:.2%}")
print(f"Accuracy on late (genuinely faulty) windows: {late_correct:.2%}")

Accuracy on early (pre-fault, mislabeled) windows: 6.20%
Accuracy on late (genuinely faulty) windows: 70.93%


In [15]:
from feature_engineering import fix_test_labels

test_fixed_for_lstm = fix_test_labels(test)
test_filtered_fixed = test_fixed_for_lstm[~test_fixed_for_lstm["faultNumber"].isin(HARD_TO_DETECT_FAULTS)]

X_test_seq_fixed, y_test_seq_fixed = build_sequences(test_filtered_fixed, measurement_cols, window=20)

print("Fixed test sequences:", X_test_seq_fixed.shape)

Relabeled 3360 rows from faulty to normal (pre-fault period in test data)
Fixed test sequences: (18359, 20, 52)


In [16]:
test_with_run_id = test.copy()
test_with_run_id["run_id"] = test_with_run_id["faultNumber"]

test_fixed_with_run_id = fix_test_labels(test_with_run_id)

test_filtered_final = test_fixed_with_run_id[
    ~test_fixed_with_run_id["run_id"].isin(HARD_TO_DETECT_FAULTS)
]

print(test_filtered_final[["run_id", "faultNumber"]].drop_duplicates().sort_values("run_id"))

Relabeled 3360 rows from faulty to normal (pre-fault period in test data)
       run_id  faultNumber
0           0            0
960         1            0
1120        1            1
1920        2            0
2080        2            2
3840        4            0
4000        4            4
4800        5            0
4960        5            5
5760        6            0
5920        6            6
6720        7            0
6880        7            7
7680        8            0
7840        8            8
9600       10            0
9760       10           10
10560      11            0
10720      11           11
11520      12            0
11680      12           12
12480      13            0
12640      13           13
13440      14            0
13600      14           14
15360      16            0
15520      16           16
16320      17            0
16480      17           17
17280      18            0
17440      18           18
18240      19            0
18400      19           19
19200   

In [17]:
def build_sequences_correct(df, feature_cols, window=20, step=1):
    X_sequences = []
    y_sequences = []
    skipped = 0

    for run_id in sorted(df["run_id"].unique()):
        run_data = df[df["run_id"] == run_id]
        values = run_data[feature_cols].values
        labels = run_data["faultNumber"].values

        for start in range(0, len(values) - window + 1, step):
            window_labels = labels[start:start + window]
            if len(set(window_labels)) == 1:
                X_sequences.append(values[start:start + window])
                y_sequences.append(window_labels[0])
            else:
                skipped += 1

    print(f"Built {len(X_sequences)} sequences, skipped {skipped} "
          f"(straddling a label change)")
    return np.array(X_sequences), np.array(y_sequences)


X_test_seq_correct, y_test_seq_correct = build_sequences_correct(
    test_filtered_final, measurement_cols, window=20
)

print("Corrected test sequences:", X_test_seq_correct.shape)

Built 17537 sequences, skipped 342 (straddling a label change)
Corrected test sequences: (17537, 20, 52)


In [18]:
n_test_correct_samples = X_test_seq_correct.shape[0]

X_test_seq_correct_scaled = scaler_clean.transform(X_test_seq_correct.reshape(-1, n_features))
X_test_seq_correct_scaled = X_test_seq_correct_scaled.reshape(n_test_correct_samples, n_timesteps, n_features)

y_test_seq_correct_encoded = label_encoder_clean.transform(y_test_seq_correct)

test_loss_correct, test_accuracy_correct = model_lstm_v3.evaluate(
    X_test_seq_correct_scaled, y_test_seq_correct_encoded, verbose=0
)

print(f"LSTM test accuracy (corrected): {test_accuracy_correct:.2%}")
#print(f"XGBoost (tuned + trend) test accuracy: 81.46%")


LSTM test accuracy (corrected): 66.87%


In [19]:
from sklearn.metrics import classification_report

y_pred_lstm_correct_encoded = model_lstm_v3.predict(X_test_seq_correct_scaled, verbose=0).argmax(axis=1)
y_pred_lstm_correct = label_encoder_clean.inverse_transform(y_pred_lstm_correct_encoded)

print(classification_report(y_test_seq_correct, y_pred_lstm_correct))


              precision    recall  f1-score   support

           0       0.70      0.40      0.51      3479
           1       0.89      1.00      0.94       781
           2       0.94      0.99      0.97       781
           4       0.89      0.96      0.92       781
           5       0.76      0.97      0.85       781
           6       1.00      1.00      1.00       781
           7       0.95      1.00      0.97       781
           8       0.66      0.46      0.54       781
          10       0.38      0.36      0.37       781
          11       0.82      0.73      0.78       781
          12       0.59      0.66      0.62       781
          13       0.91      0.25      0.39       781
          14       1.00      1.00      1.00       781
          16       0.21      0.33      0.25       781
          17       0.86      0.94      0.90       781
          18       0.57      0.88      0.69       781
          19       0.75      0.95      0.84       781
          20       0.49    

In [20]:
# Adding a dropout layer to the LSTM model to reduce overfitting.
#  The dropout layer randomly sets a fraction of input units to 0 at each update during training time, which helps prevent overfitting.

model_lstm_dropout = keras.Sequential([
    layers.Input(shape=(n_timesteps, n_features)),
    layers.LSTM(64, dropout=0.3, recurrent_dropout=0.3),
    layers.Dense(19, activation="softmax")
])

model_lstm_dropout.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)

history_dropout = model_lstm_dropout.fit(
    X_train_clean_scaled, y_train_clean_encoded,
    validation_data=(X_val_clean_scaled, y_val_clean_encoded),
    epochs=20,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.3509 - loss: 2.2875 - val_accuracy: 0.4758 - val_loss: 1.7872
Epoch 2/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.5710 - loss: 1.4750 - val_accuracy: 0.6319 - val_loss: 1.2653
Epoch 3/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6582 - loss: 1.1258 - val_accuracy: 0.7232 - val_loss: 1.0108
Epoch 4/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7176 - loss: 0.9116 - val_accuracy: 0.7410 - val_loss: 0.9212
Epoch 5/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7711 - loss: 0.7726 - val_accuracy: 0.7219 - val_loss: 0.8817
Epoch 6/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.7960 - loss: 0.6816 - val_accuracy: 0.7144 - val_loss: 0.8473
Epoch 7/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8088 - loss: 0.6166 - val_accuracy: 0.7423 - val_loss: 0.8331
Epoch 8/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8374 - loss: 0.5350 - val_acc

In [21]:
# Smaller LSTM
model_lstm_small = keras.Sequential([
    layers.Input(shape=(n_timesteps, n_features)),
    layers.LSTM(16),
    layers.Dense(19, activation="softmax")
])

model_lstm_small.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_small = model_lstm_small.fit(
    X_train_clean_scaled, y_train_clean_encoded,
    validation_data=(X_val_clean_scaled, y_val_clean_encoded),
    epochs=20,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.2746 - loss: 2.6295 - val_accuracy: 0.3667 - val_loss: 2.3628
Epoch 2/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5096 - loss: 2.0410 - val_accuracy: 0.4785 - val_loss: 1.8873
Epoch 3/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.6353 - loss: 1.5183 - val_accuracy: 0.5426 - val_loss: 1.5499
Epoch 4/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7255 - loss: 1.1662 - val_accuracy: 0.6026 - val_loss: 1.3296
Epoch 5/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7947 - loss: 0.9074 - val_accuracy: 0.6564 - val_loss: 1.1913
Epoch 6/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8351 - loss: 0.7200 - val_accuracy: 0.6660 - val_loss: 1.1264
Epoch 7/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8633 - loss: 0.5869 - val_accuracy: 0.6980 - val_loss: 1.0311
Epoch 8/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8924 - loss: 0.4846 - val_accuracy